In [18]:
import os
import os.path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from datetime import datetime
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import base64
sns.set()

In [19]:
datadir = "data"
data = os.path.join(datadir, "travelers-conveyances-fy21-fy24.csv")
df_cbp = pd.read_csv(data)
df_cbp = df_cbp.rename(columns = {"Month (abbv)": "Month"})
df_cbp['Port of Entry'] = df_cbp['Port of Entry'].str.replace(r'\s*\(\d+\)', '', regex=True)

In [20]:
df_cbp_air = df_cbp[df_cbp["Mode of Transportation"] == "Air"]

In [21]:
df_cbp_air_traveler = df_cbp_air[df_cbp_air["Measure Name"] == "Travelers"]

In [22]:
df_cbp_air_traveler.sort_values(by = ["FY", "Month", "Count"], ascending = [False, False, False]).head(20)

,FY,Month Grouping,Month,Region,Field Office,State,Port of Entry,Mode of Transportation,Measure Name,Category,Count
15840,2024,FYTD,SEP,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1556149
14378,2024,FYTD,SEP,Coastal/Interior,LOS ANGELES,CA,LOS ANGELES INT ARPT,Air,Travelers,Air,932925
19065,2024,FYTD,SEP,Coastal/Interior,MIAMI,FL,MIAMI INTL AIRPORT,Air,Travelers,Air,866843
13359,2024,FYTD,SEP,Coastal/Interior,SAN FRANCISCO,CA,SAN FRANCISCO INTL AIRPT,Air,Travelers,Air,608898
41194,2024,FYTD,SEP,Coastal/Interior,NEW YORK,NJ,NEW YORK/NEWARK AREA,Air,Travelers,Air,585601
34970,2024,FYTD,SEP,Coastal/Interior,ATLANTA,GA,"ATLANTA, GA",Air,Travelers,Air,540758
26185,2024,FYTD,SEP,Preclearance,PRECLEARANCE,DC,USCBP TORONTO PRECLEAR,Air,Travelers,Air,528660
38320,2024,FYTD,SEP,Coastal/Interior,CHICAGO,IL,"CHICAGO, IL",Air,Travelers,Air,528034
9070,2024,FYTD,SEP,Coastal/Interior,BALTIMORE,DC,"WASHINGTON, DC",Air,Travelers,Air,431269
8196,2024,FYTD,SEP,Coastal/Interior,HOUSTON,TX,HOUSTON INTERCONTL,Air,Travelers,Air,431178


In [23]:
df_cbp_air.sort_values(by = "Count", ascending = False)

,FY,Month Grouping,Month,Region,Field Office,State,Port of Entry,Mode of Transportation,Measure Name,Category,Count
33525,2024,FYTD,AUG,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1919622
13812,2024,FYTD,JUL,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1872183
47710,2023,FYTD,AUG,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1798867
7753,2023,FYTD,JUL,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1746306
17673,2024,FYTD,JUN,Coastal/Interior,NEW YORK,NY,JOHN F KENNEDY AIRPORT,Air,Travelers,Air,1571131
...,...,...,...,...,...,...,...,...,...,...,...
7749,2024,FYTD,FEB,Coastal/Interior,SAN JUAN,PR,"PONCE, PR",Air,Conveyances,Air,1
20879,2023,FYTD,AUG,Coastal/Interior,BALTIMORE,VA,"NEWPORT NEWS, VA",Air,Conveyances,Air,1
20888,2022,FYTD,DEC,Coastal/Interior,SAN FRANCISCO,HI,"HILO, HI",Air,Conveyances,Air,1
2100,2022,FYTD,MAR,Northern Border,SEATTLE,WA,"PORT TOWNSEND, WA",Air,Travelers,Air,1


In [24]:
bar_data = df_cbp[df_cbp['Measure Name'] == 'Travelers']
bar_data = bar_data.groupby(['Port of Entry', 'Region'])['Count'].sum().reset_index()

# Get the top 10 ports by total count
top_ports_bar = bar_data.nlargest(10, 'Count')

# Bar chart for top ports
fig_bar = px.bar(top_ports_bar, 
                 x='Count', 
                 y='Port of Entry', 
                 color='Region',
                 orientation='h',
                 title="Top 10 Ports of Entry by Total Traveler Count",
                 labels={'Count': 'Total Traveler Count', 'Port of Entry': 'Port of Entry'})
fig_bar.update_layout(height=600)
fig_bar.show()

In [25]:
# Filter and aggregate data for stacked bar chart by month and transportation mode
stacked_data = df_cbp[df_cbp['Measure Name'] == 'Travelers']
stacked_data = stacked_data.groupby(['FY', 'Mode of Transportation'])['Count'].sum().reset_index()

# Stacked bar chart by mode of transportation
fig_stacked = px.bar(stacked_data, 
                     x='FY', 
                     y='Count', 
                     color='Mode of Transportation',
                     title="Monthly Traveler Count by Mode of Transportation",
                     labels={'Count': 'Traveler Count', 'FY': 'Month'})
fig_stacked.update_layout(barmode='stack', height=600)
fig_stacked.show()

In [26]:
# Prepare data for line or area chart by month and region
line_data = df_cbp[df_cbp['Measure Name'] == 'Travelers']
line_data['Date'] = pd.to_datetime(line_data['FY'].astype(str) + '-' + line_data['Month'], format='%Y-%b')
line_data = line_data.groupby(['Date', 'Region'])['Count'].sum().reset_index()

# Line chart by region
fig_line = px.line(line_data, 
                   x='Date', 
                   y='Count', 
                   color='Region',
                   title="Monthly Traveler Trends by Region",
                   labels={'Count': 'Traveler Count', 'Date': 'Date'})
fig_line.update_layout(height=600)

# Optional: Uncomment for an area chart instead of a line chart
# fig_line.update_traces(mode='lines+markers', stackgroup='one')

fig_line.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_67404\881643699.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [27]:
# Prepare data for line or area chart by month and region
line_data = df_cbp[df_cbp['Measure Name'] == 'Travelers']
line_data['Date'] = pd.to_datetime(line_data['FY'].astype(str) + '-' + line_data['Month'], format='%Y-%b')
line_data = line_data.groupby(['Date', 'Category'])['Count'].sum().reset_index()

# Line chart by region
fig_line = px.line(line_data, 
                   x='Date', 
                   y='Count', 
                   color='Category',
                   title="Monthly Traveler Trends by Mode of Transportation",
                   labels={'Count': 'Traveler Count', 'Date': 'Date'})
fig_line.update_layout(height=600)

# Optional: Uncomment for an area chart instead of a line chart
# fig_line.update_traces(mode='lines+markers', stackgroup='one')

fig_line.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_67404\2387306057.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [28]:
treemap_data = df_cbp[df_cbp['Measure Name'] == 'Travelers'] 

fig_treemap = px.treemap(treemap_data, 
                         path=['Region', 'State', 'Port of Entry'], 
                         values='Count', 
                         color='Count',
                         color_continuous_scale='pubu')

with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_treemap.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1.025,
        sizex=0.15, sizey=0.15,
        xanchor="right", yanchor="bottom"
    )
)

fig_treemap.update_layout(
    margin=dict(t=110, l=25, r=25, b=25), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Distribution of Travelers by Region, State, Port of Entry 2021 - 2024</b>",
        font = dict(
            size=26,
            family = "Helvetica")),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Traveler and Conveyance Statistics</i>", 
            x=0.036,  
            y=1.064,  
            font=dict(size=16, family="Helvetica"),  
            showarrow=False  
        )
    ]
)

fig_treemap.write_html("interactive_plots/cbp_distribution.html")

In [29]:
treemap_data = df_cbp[df_cbp['Measure Name'] == 'Travelers'].groupby(['State', 'Port of Entry']).sum()
treemap_data.reset_index(inplace=True)

fig_treemap = px.treemap(treemap_data, 
                         path=['State', 'Port of Entry'], 
                         values='Count', 
                         color='Count',
                         color_continuous_scale='pubu')

with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_treemap.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1.025,
        sizex=0.15, sizey=0.15,
        xanchor="right", yanchor="bottom"
    )
)

fig_treemap.update_layout(
    margin=dict(t=110, l=25, r=25, b=25), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Distribution of Travelers by State and Port of Entry 2021 - 2024</b>",
        font = dict(
            size=28,
            family = "Helvetica")),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Traveler and Conveyance Statistics</i>",  # Replace with your actual data source
            x=0.036,  # Center the annotation
            y=1.064,  # Position slightly above the main title
            font=dict(size=16, family="Helvetica"),  # Font size for the subtitle
            showarrow=False  # No arrow for the subtitle
        )
    ]
)

fig_treemap.write_html("interactive_plots/cbp_state_distribution.html")

In [35]:
sunburst_airport_data = df_cbp[(df_cbp['Measure Name'] == 'Travelers') & (df_cbp['Category'] == 'Air')].groupby(['State', 'Port of Entry']).sum()
sunburst_airport_data.reset_index(inplace=True)
sunburst_airport_data.sort_values(by = "Count", ascending = False, inplace = True)
category_order = sunburst_airport_data.groupby("State").sum().sort_values(by="Count", ascending=False).reset_index()["State"]

fig_sunburst = px.treemap(sunburst_airport_data, 
                         path=['State', 'Port of Entry'], 
                         values = 'Count',
                         custom_data = ["State", "Port of Entry", "Count"],
                         color = "State", 
                         color_discrete_sequence = ["#012A4A", "#013A63", "#01497C", "#014F86", "#2A6F97", "#2C7DA0", "#468FAF", "#61A5C2", "#89C2D9", "#A9D6E5"],
                         color_discrete_map = {
                            'DC': "#012A4A",    #1
                            'FL': "#013A63",
                            'CA': "#01497C",
                            'NY': "#014F86",
                            'TX': "#2A6F97",    #5
                            'NJ': "#2C7DA0",
                            'IL': "#468FAF",
                            'GA': "#61A5C2",
                            'MA': "#89C2D9",
                            'WA': "#A9D6E5",    #10
                            'NC': "#012A4A",
                            'CO': "#013A63",
                            'PA': "#01497C",
                            'MI': "#014F86",
                            'HI': "#2A6F97",    #15
                            'VI': "#2C7DA0",
                            'MN': "#468FAF",
                            'AZ': "#61A5C2",
                            'NV': "#89C2D9",
                            'GU': "A9D6E5"
                         }
                         )

with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_sunburst.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1.025,
        sizex=0.2, sizey=0.2,
        xanchor="right", yanchor="bottom"
    )
)

fig_sunburst.update_traces(
    #insidetextorientation = 'radial',
    hovertemplate =
                "<b>Port of Entry:</b><br>" +
                "<b>%{customdata[1]}</b><br><br>" +
                "State: %{customdata[0]}<br>" +
                "Number of Travelers: %{customdata[2]}<br>" +
                "<extra></extra>")

fig_sunburst.update_layout(
    margin=dict(t=150, l=25, r=25, b=25), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Inbound International Travelers at U.S. Airports 2021 - 2024</b>",
        font = dict(
            size=28,
            family = "Helvetica"),
        x = 0.018),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Traveler and Conveyance Statistics</i>",  # Replace with your actual data source
            #x=0.036,  # Center the annotation
            x = 0,
            y=1.09,  # Position slightly above the main title
            font=dict(size=16, family="Helvetica"),  # Font size for the subtitle
            showarrow=False  # No arrow for the subtitle
        )
    ]
)

fig_sunburst.write_html("interactive_plots/cbp_airport_distribution.html")

In [31]:
sunburst_data = df_cbp[df_cbp['Measure Name'] == 'Travelers'].groupby(['State', 'Port of Entry']).sum()
sunburst_data.reset_index(inplace=True)

fig_sunburst = px.sunburst(sunburst_data, 
                         path=['State', 'Port of Entry'], 
                         values='Count',
                         color = "State", 
                         color_discrete_sequence = ["#012A4A", "#013A63", "#01497C", "#014F86", "#2A6F97", "#2C7DA0", "#468FAF", "#61A5C2", "#89C2D9", "#A9D6E5"])


with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_sunburst.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1.025,
        sizex=0.15, sizey=0.15,
        xanchor="right", yanchor="bottom"
    )
)

fig_sunburst.update_traces(insidetextorientation = 'radial')

fig_sunburst.update_layout(
    margin=dict(t=110, l=25, r=25, b=25), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Distribution of Travelers by State and Port of Entry 2021 - 2024</b>",
        font = dict(
            size=28,
            family = "Helvetica")),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Traveler and Conveyance Statistics</i>",  # Replace with your actual data source
            x=0.036,  # Center the annotation
            y=1.064,  # Position slightly above the main title
            font=dict(size=16, family="Helvetica"),  # Font size for the subtitle
            showarrow=False  # No arrow for the subtitle
        )
    ]
)

fig_sunburst.write_html("interactive_plots/cbp_state_distribution_sunburst.html")

In [32]:
df = px.data.tips()

In [33]:
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [34]:
line_chart_data = df_cbp[df_cbp['Measure Name'] == 'Travelers']  # Filter only for travelers
line_chart_data['Date'] = pd.to_datetime(line_chart_data['FY'].astype(str) + '-' + line_chart_data['Month'], format='%Y-%b')
line_chart_data = line_chart_data.groupby(['Date'])['Count'].sum().reset_index()
fig_line = px.line(line_chart_data, x='Date', y='Count', title="Seasonal Trend of Traveler Entries")
fig_line.update_traces(line=dict(color='firebrick'))
fig_line.update_layout(xaxis_title="Date", yaxis_title="Traveler Count")


C:\Users\Admin\AppData\Local\Temp\ipykernel_67404\2168285421.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

